In [ ]:
# Cell 1: GitHub Sync & Setup
import os
from google.colab import userdata

# --- Configuration for Git ---
GITHUB_USER = "MagicMorgigi"
GITHUB_REPO = "YourRepoName"
BRANCH = "main"
USER_EMAIL = "fmorgalla@yahoo.de"
USER_NAME = "FM"

# Retrieve PAT from Colab Secrets (ensure you have a secret named 'github_pat')
try:
    GITHUB_PAT = userdata.get('MA_colab_github_PAT')
except Exception as e:
    print("Error: 'github_pat' not found in Colab Secrets. Please add it.")
    GITHUB_PAT = None

REPO_URL = f"https://{GITHUB_PAT}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
WORKSPACE_DIR = f"/content/{GITHUB_REPO}"

if GITHUB_PAT:
    os.system(f"git config --global user.email '{USER_EMAIL}'")
    os.system(f"git config --global user.name '{USER_NAME}'")

    if not os.path.exists(WORKSPACE_DIR):
        print(f"Cloning repository {GITHUB_REPO}...")
        !git clone {REPO_URL}
    else:
        print(f"Repository {GITHUB_REPO} already exists. Pulling latest changes...")
        %cd {WORKSPACE_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
        %cd /content

    # --- Create / Update .gitignore ---
    gitignore_content = """
    # Ignore Colab temporary files
    __pycache__/
    *.pyc
    .ipynb_checkpoints/

    # Ignore data files and archives (handled via GDrive)
    *.tar.gz
    *.pt
    *.tif

    # Ignore local extractions
    /data_local/

    # Track only Notebooks, READMEs, and configurations
    !*.ipynb
    !README.md
    !.gitignore
    """.strip()

    with open(os.path.join(WORKSPACE_DIR, ".gitignore"), "w") as f:
        f.write(gitignore_content)

    print("Git setup complete. .gitignore created.")

In [ ]:
# Cell 2: Imports and Seeding
import os
import re
import math
import shutil
import tarfile
import random
import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# Ensure Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed = 42
seed_everything(seed)
print(f"PyTorch version: {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Cell 3: Configuration
@dataclass(frozen=True)
class PipelineConfig:
    # Google Drive Paths (Update these with your actual Drive paths)
    drive_base: str = "/content/drive/MyDrive/master_thesis_FM/Datasets/BioMassters/packed_compressed_data/tar_gz_archives"

    # Archive paths
    train_feat_archive: str = f"{drive_base}/embedded_features/6%_sampled/train.tar.gz"
    train_label_archive: str = f"{drive_base}/labels/6%_sampled/train.tar.gz"
    val_feat_archive: str = f"{drive_base}/embedded_features/6%_sampled/val.tar.gz"
    val_label_archive: str = f"{drive_base}/labels/6%_sampled/val.tar.gz"
    test_feat_archive: str = f"{drive_base}/embedded_features/6%_sampled/test.tar.gz"
    test_label_archive: str = f"{drive_base}/labels/6%_sampled/test.tar.gz"

    # Local Storage Paths (Fast I/O)
    local_data_dir: str = "/content/data_local"

    # Hyperparameters
    batch_size: int = 16
    learning_rate: float = 1e-3
    epochs: int = 50
    patience: int = 15 # For early stopping
    num_workers: int = 4

    # Data params
    feature_dim: int = 1536
    img_size: int = 256


    exp_name = f"AS_LP_6%_subset_BS{batch_size}_LR{str(learning_rate).split('.')[1]}}"

    # Output Paths
    output_dir: str = f"/content/drive/MyDrive/Colab_Notebooks/AnySat/linear_probing/experiments/{exp_name}"
    checkpoint_path: str = output_dir
    stats_path: str = output_dir

config = PipelineConfig()
os.makedirs(config.output_dir, exist_ok=True)

In [ ]:
# Cell 4: Mount Drive & Sequential Extraction
from google.colab import drive
drive.mount('/content/drive')

def process_archive_sequentially(drive_path: str, split_name: str, data_type: str):
    """Copies, extracts, and deletes an archive to avoid storage spikes."""
    local_dest = os.path.join(config.local_data_dir, split_name, data_type)
    if os.path.exists(local_dest) and len(os.listdir(local_dest)) > 0:
        print(f"Skipping {split_name} {data_type} (already extracted).")
        return local_dest

    os.makedirs(local_dest, exist_ok=True)
    temp_tar = os.path.join(config.local_data_dir, f"temp_{split_name}_{data_type}.tar.gz")

    print(f"Copying {drive_path} to local...")
    shutil.copy2(drive_path, temp_tar)

    print(f"Extracting {temp_tar}...")
    with tarfile.open(temp_tar, "r:gz") as tar:
        tar.extractall(path=local_dest)

    print(f"Deleting temporary archive {temp_tar}...\n")
    os.remove(temp_tar)
    return local_dest

# Process sequentially
paths = {
    "train": {"feat": config.train_feat_archive, "label": config.train_label_archive},
    "val": {"feat": config.val_feat_archive, "label": config.val_label_archive},
    "test": {"feat": config.test_feat_archive, "label": config.test_label_archive},
}

local_paths = {}
for split in ["train", "val", "test"]:
    local_paths[split] = {}
    local_paths[split]["feat"] = process_archive_sequentially(paths[split]["feat"], split, "features")
    local_paths[split]["label"] = process_archive_sequentially(paths[split]["label"], split, "labels")

print("All archives successfully extracted to Colab local storage.")

In [ ]:
# Cell 5: Dataset & DataLoader
class AGBDataset(Dataset):
    def __init__(self, feat_dir, label_dir):
        self.feat_dir = Path(feat_dir)
        self.label_dir = Path(label_dir)

        # Regex to find the 8-character alphanumeric ID
        self.id_pattern = re.compile(r'([A-Za-z0-9]{8})')

        self.samples = self._pair_files()

    def _pair_files(self):
        # Support both .pt and .tif extensions for features as mentioned in prompt
        feat_files = list(self.feat_dir.glob("**/*.pt")) + list(self.feat_dir.glob("**/*.tif"))
        label_files = list(self.label_dir.glob("**/*.tif"))

        feat_dict = {self._extract_id(f.name): f for f in feat_files if self._extract_id(f.name)}
        label_dict = {self._extract_id(f.name): f for f in label_files if self._extract_id(f.name)}

        samples = []
        for file_id, feat_path in feat_dict.items():
            if file_id in label_dict:
                samples.append((feat_path, label_dict[file_id], file_id))

        print(f"Found {len(samples)} valid pairs in {self.feat_dir.parent.name}")
        return samples

    def _extract_id(self, filename):
        match = self.id_pattern.search(filename)
        return match.group(1) if match else None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat_path, label_path, file_id = self.samples[idx]

        # Load Features: Handle both .pt and .tif formats
        if feat_path.suffix == '.pt':
            feat = torch.load(feat_path, map_location='cpu') # [256, 256, 1536], float16
        else:
            feat = torch.from_numpy(tifffile.imread(feat_path))

        # Rearrange to [C, H, W] for PyTorch convolutions and cast to float32
        feat = feat.permute(2, 0, 1).float()

        # Load Labels: [256, 256] -> [1, 256, 256]
        label = tifffile.imread(label_path).astype(np.float32)
        label = torch.from_numpy(label).unsqueeze(0)

        return feat, label, file_id

# Instantiate DataLoaders
train_dataset = AGBDataset(local_paths["train"]["feat"], local_paths["train"]["label"])
val_dataset = AGBDataset(local_paths["val"]["feat"], local_paths["val"]["label"])

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                          num_workers=config.num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                        num_workers=config.num_workers, pin_memory=True)

In [ ]:
# Cell 6: Model Architecture & Standardization
class LinearProbe(nn.Module):
    def __init__(self, in_features, out_features=1):
        super().__init__()
        # 1x1 Conv acts as an independent linear layer per pixel
        self.probe = nn.Conv2d(in_features, out_features, kernel_size=1)

    def forward(self, x):
        return self.probe(x)

def get_standardization_stats(loader, stats_path):
    if os.path.exists(stats_path):
        print(f"Loading existing stats from {stats_path}")
        return torch.load(stats_path)

    print("Calculating feature statistics (mean and std) over training set...")
    mean = torch.zeros(config.feature_dim)
    var = torch.zeros(config.feature_dim)
    pixel_count = 0

    # Online calculation to avoid RAM overflow
    for feats, _, _ in tqdm(loader, desc="Standardization"):
        # feats is [B, 1536, 256, 256]
        b, c, h, w = feats.shape
        feats_flat = feats.transpose(0, 1).reshape(c, -1) # [1536, B*H*W]

        batch_mean = feats_flat.mean(dim=1)
        batch_var = feats_flat.var(dim=1, unbiased=False)
        batch_pixels = feats_flat.shape[1]

        # Combine batches (Welford's algorithm simplified)
        delta = batch_mean - mean
        tot_pixels = pixel_count + batch_pixels

        mean = mean + delta * batch_pixels / tot_pixels
        var = (var * pixel_count + batch_var * batch_pixels + (delta**2) * pixel_count * batch_pixels / tot_pixels) / tot_pixels
        pixel_count = tot_pixels

    std = torch.sqrt(var + 1e-7)
    stats = {'mean': mean.view(1, -1, 1, 1), 'std': std.view(1, -1, 1, 1)}
    torch.save(stats, stats_path)
    return stats

# Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LinearProbe(config.feature_dim).to(device)
stats = get_standardization_stats(train_loader, config.stats_path)
stats['mean'] = stats['mean'].to(device)
stats['std'] = stats['std'].to(device)

In [ ]:
# Cell 7: Global Metrics Tracker
class MetricsTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum_sq_error = 0.0
        self.sum_abs_error = 0.0
        self.sum_target = 0.0
        self.sum_sq_target = 0.0
        self.pixel_count = 0

    @torch.no_grad()
    def update(self, preds, targets):
        """Updates running sums. Valid for batch sizes and dimensions."""
        preds = preds.detach().view(-1).float()
        targets = targets.detach().view(-1).float()

        # Mask out NaNs if any exist in the LiDAR targets
        mask = ~torch.isnan(targets)
        p, t = preds[mask], targets[mask]

        self.sum_sq_error += torch.sum((p - t)**2).item()
        self.sum_abs_error += torch.sum(torch.abs(p - t)).item()
        self.sum_target += torch.sum(t).item()
        self.sum_sq_target += torch.sum(t**2).item()
        self.pixel_count += t.numel()

    def compute(self):
        if self.pixel_count == 0: return 0, 0, 0, 0

        mse = self.sum_sq_error / self.pixel_count
        rmse = math.sqrt(mse)
        mae = self.sum_abs_error / self.pixel_count

        # Exact Global R2 via Sum of Squares
        mean_target = self.sum_target / self.pixel_count
        ss_tot = self.sum_sq_target - (self.sum_target**2 / self.pixel_count)

        r2 = 1 - (self.sum_sq_error / ss_tot) if ss_tot > 0 else 0.0
        return mse, rmse, mae, r2

In [ ]:
# Cell 8: Training Loop
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
criterion = nn.MSELoss()
scaler = GradScaler() # For mixed precision

history = {"train_mse": [], "val_mse": [], "val_rmse": [], "val_mae": [], "val_r2": []}
start_epoch = 0
best_val_loss = float('inf')
epochs_no_improve = 0

# --- Resume Functionality ---
if os.path.exists(config.checkpoint_path):
    print(f"Found checkpoint at {config.checkpoint_path}. Resuming training...")
    checkpoint = torch.load(config.checkpoint_path)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    history = pd.read_csv(os.path.join(config.output_dir, "training_history.csv")).to_dict(orient="list")

for epoch in range(start_epoch, config.epochs):
    # Training Phase
    model.train()
    train_tracker = MetricsTracker()

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Train]")
    for feats, targets, _ in train_pbar:
        feats, targets = feats.to(device), targets.to(device)
        feats = (feats - stats['mean']) / stats['std'] # Apply standardization

        optimizer.zero_grad()
        with autocast():
            preds = model(feats)
            loss = criterion(preds, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_tracker.update(preds, targets)
        train_pbar.set_postfix(mse=loss.item())

    train_mse, _, _, _ = train_tracker.compute()

    # Validation Phase
    model.eval()
    val_tracker = MetricsTracker()

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Val]")
    for feats, targets, _ in val_pbar:
        feats, targets = feats.to(device), targets.to(device)
        feats = (feats - stats['mean']) / stats['std']

        with torch.no_grad():
            with autocast():
                preds = model(feats)

        val_tracker.update(preds, targets)

    val_mse, val_rmse, val_mae, val_r2 = val_tracker.compute()

    # Save History
    history["train_mse"].append(train_mse)
    history["val_mse"].append(val_mse)
    history["val_rmse"].append(val_rmse)
    history["val_mae"].append(val_mae)
    history["val_r2"].append(val_r2)
    pd.DataFrame(history).to_csv(os.path.join(config.output_dir, "training_history.csv"), index=False)

    print(f"Epoch {epoch+1}: Train MSE={train_mse:.4f} | Val MSE={val_mse:.4f} | Val R2={val_r2:.4f}")

    # Early Stopping & Checkpointing
    if val_mse < best_val_loss:
        best_val_loss = val_mse
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_val_loss': best_val_loss
        }, config.checkpoint_path)
        print("-> Checkpoint saved!")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= config.patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

In [ ]:
# Cell 9: Plot Training Curves
history_df = pd.read_csv(os.path.join(config.output_dir, "training_history.csv"))

plt.figure(figsize=(10, 6))
plt.plot(history_df['train_mse'], label='Train MSE', linewidth=2)
plt.plot(history_df['val_mse'], label='Validation MSE', linewidth=2)
plt.title('Training and Validation Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plot_path = os.path.join(config.output_dir, "loss_curve.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Curve saved to {plot_path}")

In [ ]:
# Cell 10: Inference & Visualization
import matplotlib.colors as mcolors

# Ensure test dataset exists
test_dataset = AGBDataset(local_paths["test"]["feat"], local_paths["test"]["label"])
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

# Load best model & stats
inf_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = LinearProbe(config.feature_dim).to(inf_device)
checkpoint = torch.load(config.checkpoint_path, map_location=inf_device)
best_model.load_state_dict(checkpoint['model_state'])
best_model.eval()

inf_stats = torch.load(config.stats_path, map_location=inf_device)

test_tracker = MetricsTracker()
predictions_cache = [] # Cache a few for plotting

print("Running inference on test set...")
for i, (feats, targets, ids) in enumerate(tqdm(test_loader, desc="Testing")):
    feats, targets = feats.to(inf_device), targets.to(inf_device)
    feats = (feats - inf_stats['mean']) / inf_stats['std']

    with torch.no_grad():
        with autocast():
            preds = best_model(feats)

    test_tracker.update(preds, targets)

    # Save first batch for visualization
    if i == 0:
        predictions_cache = [(ids[b], preds[b, 0].cpu().numpy(), targets[b, 0].cpu().numpy()) for b in range(min(5, len(ids)))]

test_mse, test_rmse, test_mae, test_r2 = test_tracker.compute()

# Save Metrics to TXT
metrics_path = os.path.join(config.output_dir, "final_metrics.txt")
with open(metrics_path, "w") as f:
    f.write(f"--- Best Model Validation Metrics ---\n")
    f.write(f"Best Val MSE:  {checkpoint['best_val_loss']:.4f}\n\n")
    f.write(f"--- Test Set Metrics ---\n")
    f.write(f"Test MSE:  {test_mse:.4f}\n")
    f.write(f"Test RMSE: {test_rmse:.4f}\n")
    f.write(f"Test MAE:  {test_mae:.4f}\n")
    f.write(f"Test R2:   {test_r2:.4f}\n")
print(f"Metrics saved to {metrics_path}")

# Visualization
num_vis = len(predictions_cache)
fig, axes = plt.subplots(num_vis, 2, figsize=(12, 4 * num_vis))
fig.suptitle('AGB Estimation: Prediction vs Target', fontsize=16)

# Normalize colors across pairs for fair comparison
for idx, (img_id, pred, target) in enumerate(predictions_cache):
    vmin = min(pred.min(), target.min())
    vmax = max(pred.max(), target.max())
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    # Target
    im0 = axes[idx, 0].imshow(target, cmap='viridis', norm=norm)
    axes[idx, 0].set_title(f"Target AGB (ID: {img_id})")
    axes[idx, 0].axis('off')

    # Prediction
    im1 = axes[idx, 1].imshow(pred, cmap='viridis', norm=norm)
    axes[idx, 1].set_title(f"Predicted AGB (ID: {img_id})")
    axes[idx, 1].axis('off')

    fig.colorbar(im0, ax=axes[idx, :], orientation='vertical', fraction=0.02, pad=0.04)

vis_path = os.path.join(config.output_dir, "inference_comparison.png")
plt.tight_layout()
plt.savefig(vis_path, dpi=300)
plt.show()
print(f"Visualization saved to {vis_path}")

In [ ]:
# Cell 11: Commit and Push to GitHub
%cd {WORKSPACE_DIR}

# Copy the notebook into the workspace if it's currently in /content
# Note: You must manually save the notebook in the Colab UI (Ctrl+S) before running this.
NOTEBOOK_NAME = "AGB_Linear_Probing.ipynb" # Change to your actual notebook name
if os.path.exists(f"/content/{NOTEBOOK_NAME}"):
    shutil.copy(f"/content/{NOTEBOOK_NAME}", f"{WORKSPACE_DIR}/{NOTEBOOK_NAME}")

!git add .
!git commit -m "Auto-commit: Training completed and evaluated"
!git push origin {BRANCH}

%cd /content
print("Successfully pushed to GitHub.")

In [ ]:
# Cell 12: Disconnect Runtime
from google.colab import runtime
import time

print("Pipeline finished successfully. Disconnecting runtime in 5 seconds to save resources...")
time.sleep(5)
runtime.unassign()